In [1]:
from keras.applications import vgg16

img_rows = 100
img_cols = 100 

model =   vgg16.VGG16(weights = 'imagenet', 
                     include_top = False, 
                     input_shape = (img_rows, img_cols, 3))

I0000 00:00:1789179685.892391      40 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789179685.955025      40 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789179687.722419      40 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


E0000 00:00:1789179689.815783      40 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 63s 1us/step


In [2]:
for layer in model.layers:
    layer.trainable = False
 
for (i,layer) in enumerate(model.layers):
    print(str(i) + " "+ layer.__class__.__name__, layer.trainable)

0 InputLayer False
1 Conv2D False
2 Conv2D False
3 MaxPooling2D False
4 Conv2D False
5 Conv2D False
6 MaxPooling2D False
7 Conv2D False
8 Conv2D False
9 Conv2D False
10 MaxPooling2D False
11 Conv2D False
12 Conv2D False
13 Conv2D False
14 MaxPooling2D False
15 Conv2D False
16 Conv2D False
17 Conv2D False
18 MaxPooling2D False


In [3]:
def addTopModel(bottom_model, num_classes, D=256):
    """creates the top or head of the model that will be 
    placed ontop of the bottom layers"""
    top_model = bottom_model.output
    top_model = Flatten(name = "flatten")(top_model)
    # top_model = GlobalAveragePooling2D()(top_model)
    top_model = Dense(D, activation = "relu")(top_model)
    top_model = Dropout(0.3)(top_model)
    top_model = Dense(num_classes, activation = "softmax")(top_model)
    return top_model

In [4]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D, ZeroPadding2D
#from keras.layers.normalization import BatchNormalization
from keras.models import Model

num_classes = 436

FC_Head = addTopModel(model, num_classes)

modelnew = Model(inputs=model.input, outputs=FC_Head)

print(modelnew.summary())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 100, 100, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 100, 100, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 50, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 50, 50, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 50, 50, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 25, 25, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 25, 25, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 25, 25, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 25, 25, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 12, 12, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 12, 12, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 12, 12, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 6, 6, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 436)            │       112,052 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,006,644 (61.06 MB)

 Trainable params: 1,291,956 (4.93 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

None


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator




train_data_dir = "../data/fruits_veg_data/Training"
validation_data_dir = "../data/fruits_veg_data/Validation"
print("Training:", len(train_data_dir))
print("Validation:", len(validation_data_dir))

# class_names = sorted(
#     set(train_data_dir["label"])
# )

# print(class_names)
# print(len(class_names))

# # Create class -> integer mapping
# class_to_index = {
#     class_name: index
#     for index, class_name in enumerate(class_names)
# }

train_datagen = ImageDataGenerator(
      rescale=1./255,
      rotation_range=20,
      width_shift_range=0.2,
      height_shift_range=0.2,
      horizontal_flip=True,
      fill_mode='nearest')
 
validation_datagen = ImageDataGenerator(rescale=1./255)
 
train_batchsize = 64
val_batchsize = 64
 
train_batchsize = 64
val_batchsize = 64

train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=train_batchsize,
    class_mode="categorical",
    shuffle=True
)

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(img_rows, img_cols),
    batch_size=val_batchsize,
    class_mode="categorical",
    shuffle=False
)

num_classes = len(train_generator.class_indices)

class_to_index = train_generator.class_indices

class_names = list(class_to_index.keys())



print()
print("========================================")
print("Dataset Information")
print("========================================")
print("Training images:", train_generator.samples)
print("Validation images:", validation_generator.samples)
print("Number of classes:", num_classes)

print()
print("Class mapping:")
print(class_to_index)
import json

CLASS_NAMES_PATH = "../ML_models/class_names.json"

with open(CLASS_NAMES_PATH, "w") as file:
    json.dump(class_names, file, indent=4)

print(f"Class names saved to: {CLASS_NAMES_PATH}")

Training: 32
Validation: 34
Found 212076 images belonging to 436 classes.
Found 186411 images belonging to 436 classes.

Dataset Information
Training images: 212076
Validation images: 186411
Number of classes: 436

Class mapping:
{'Almonds 1': 0, 'Apple 10': 1, 'Apple 11': 2, 'Apple 12': 3, 'Apple 13': 4, 'Apple 14': 5, 'Apple 17': 6, 'Apple 18': 7, 'Apple 19': 8, 'Apple 20': 9, 'Apple 21': 10, 'Apple 22': 11, 'Apple 23': 12, 'Apple 5': 13, 'Apple 6': 14, 'Apple 7': 15, 'Apple 8': 16, 'Apple 9': 17, 'Apple Braeburn 1': 18, 'Apple Red Yellow 2': 19, 'Avocado Black 1': 20, 'Avocado Black 2': 21, 'Avocado Green 1': 22, 'Banana 3': 23, 'Banana 4': 24, 'Bean pod 1': 25, 'Blackberry 1': 26, 'Blackberry 2': 27, 'Blackberry 3': 28, 'Blackberry 4': 29, 'Blackberry 5': 30, 'Cabbage red 1': 31, 'Cactus fruit green 1': 32, 'Cactus fruit red 1': 33, 'Caju seed 1': 34, 'Cantaloupe 3': 35, 'Carambola 2': 36, 'Carambola 3': 37, 'Carrot 1': 38, 'Celery 1': 39, 'Cherimoya 1': 40, 'Cherry 3': 41, 'Cherry

In [ ]:
modelnew.compile(loss = 'categorical_crossentropy',
              optimizer =  "adam"  ,
              metrics = ['accuracy'])

nb_train_samples = len(train_data_dir)
nb_validation_samples = len(validation_data_dir)
epochs = 42
batch_size = 64

from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "/app/ML_models/checkpoints/fruitveg_{epoch:03d}.keras",
    save_freq="epoch",
    save_best_only=False
)

history = modelnew.fit(
    train_generator,
    # steps_per_epoch = nb_train_samples // batch_size,
    epochs = epochs,
    # callbacks = callbacks,
    validation_data = validation_generator,
    # validation_steps = nb_validation_samples // batch_size)
)

modelnew.save("../ML_models/fruits_veg_identify.keras")

Epoch 1/42


I0000 00:00:1789179790.318518      40 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


1210/3314 ━━━━━━━━━━━━━━━━━━━━ 45:09 1s/step - accuracy: 0.0811 - loss: 4.6360

/usr/local/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


2801/3314 ━━━━━━━━━━━━━━━━━━━━ 10:53 1s/step - accuracy: 0.1374 - loss: 4.1272

/usr/local/lib/python3.11/site-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7779s 2s/step - accuracy: 0.1495 - loss: 4.0344 - val_accuracy: 0.1869 - val_loss: 4.4262
Epoch 2/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7858s 2s/step - accuracy: 0.2386 - loss: 3.3797 - val_accuracy: 0.2151 - val_loss: 4.6527
Epoch 3/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7676s 2s/step - accuracy: 0.2608 - loss: 3.2618 - val_accuracy: 0.2253 - val_loss: 4.7557
Epoch 4/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7654s 2s/step - accuracy: 0.2715 - loss: 3.1946 - val_accuracy: 0.2271 - val_loss: 4.8191
Epoch 5/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7598s 2s/step - accuracy: 0.2782 - loss: 3.1637 - val_accuracy: 0.2390 - val_loss: 4.9625
Epoch 6/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7637s 2s/step - accuracy: 0.2826 - loss: 3.1409 - val_accuracy: 0.2390 - val_loss: 4.9433
Epoch 7/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7645s 2s/step - accuracy: 0.2881 - loss: 3.1198 - val_accuracy: 0.2416 - val_loss: 5.1411
Epoch 8/42
3314/3314 ━━━━━━━━━━━━━━━━━━━━ 7676s 2s/step - accuracy: 0.2888 - loss: 3.10

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image


def predict_single_image(img_path: str):
    """Load, display, and predict a single image."""

    print(f"\n--- Evaluation Request for: {img_path} ---")

    # Check if file exists
    if not os.path.exists(img_path):
        print(f"Error: Target path '{img_path}' does not exist.")
        return
    IMG_SIZE =(img_rows,img_cols)
    # Load image
    img = image.load_img(
        img_path,
        target_size=IMG_SIZE
    )

    # Display image
    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # Convert image to NumPy array
    img_array = image.img_to_array(img)

    # Rescale exactly like the training generator
    img_array = img_array / 255.0

    # Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)

    # Make prediction
    predictions = modelnew.predict(img_batch, verbose=0)[0]

    # Get best prediction
    best_match_idx = np.argmax(predictions)
    confidence_score = predictions[best_match_idx]

    # Get class name
    predicted_class = class_names[best_match_idx]

    print()
    print("========================================")
    print("Prediction Result")
    print("========================================")
    print(
        f"Identified as: '{predicted_class}'"
    )
    print(
        f"Confidence: {confidence_score * 100:.2f}%"
    )

    # Display top 10 predictions
    top_n = 10

    top_indices = np.argsort(predictions)[-top_n:][::-1]

    print()
    print("Top Predictions")
    print("----------------------------------------")

    for rank, idx in enumerate(top_indices, start=1):
        print(
            f"{rank}. {class_names[idx]}: "
            f"{predictions[idx] * 100:.2f}%"
        )


# Test prediction
sample_test_path = "./banana1.jpg"

predict_single_image(sample_test_path)